# Triforce — Enterprise AI Inference at Scale

```
            ▲
           / \
          / RED \
         / HAT   \
        / COURAGE \
       /───────────\
      / \         / \
     / INTEL \   / IBM  \
    / POWER   \ / WISDOM \
   /───────────X───────────\
```

**Power** (Intel) · **Wisdom** (IBM) · **Courage** (Red Hat)

This notebook demonstrates multi-agent AI inference on Intel Xeon 6 CPU — classification, NER, fraud scoring, and summarization with zero GPU.

**Requirements:** Backend stack running (`make up` from project root)

In [ ]:
import httpx, json, time, sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

HEALTHCARE = 'http://localhost:8081'
FINSERV = 'http://localhost:8082'
ORCHESTRATOR = 'http://localhost:8083'

# Verify backend is running
for name, url in [('Healthcare', HEALTHCARE), ('FinServ', FINSERV), ('Orchestrator', ORCHESTRATOR)]:
    try:
        r = httpx.get(f'{url}/health', timeout=3.0)
        print(f'  ✓ {name}: {r.json()["status"]}')
    except:
        print(f'  ✗ {name}: not reachable')

## 1. Agent Discovery via A2A Protocol

The orchestrator discovers agents automatically using the A2A protocol. Each agent exposes a `/.well-known/agent-card.json` endpoint with its capabilities.

In [ ]:
agents = httpx.get(f'{ORCHESTRATOR}/api/v1/agents').json()['agents']
for agent in agents:
    print(f'\n● {agent["name"]} ({agent["status"]})')
    print(f'  URL: {agent["url"]}')
    for skill in agent['skills']:
        print(f'  ⚙ {skill["name"]} — {skill["description"]}')

## 2. Clinical NLP on Intel Xeon 6

The healthcare agent runs a 4-node LangGraph pipeline entirely on CPU:

| Step | Model | Hardware | Purpose |
|------|-------|----------|--------|
| Classify | granite-4-0-h-tiny | Xeon 6 | Document type classification |
| Extract Entities | granite-4-0-h-tiny | Xeon 6 | Medical NER |
| Check Interactions | MCP Tool | — | Drug-drug interaction screen |
| Summarize | granite-3-2-8b-instruct | Xeon 6 | Clinical summary |

In [ ]:
clinical_text = """DISCHARGE SUMMARY: 72-year-old male with Type 2 Diabetes (E11.9),
Hypertension (I10), and CAD (I25.10). Medications: Metformin 500mg BID,
Lisinopril 10mg daily, Atorvastatin 80mg, Aspirin 81mg daily.
Recent inferior STEMI with PCI to RCA. HbA1c 7.2, Creatinine 1.4.
Discharged stable. Follow-up cardiology 2 weeks."""

start = time.monotonic()
result = httpx.post(f'{HEALTHCARE}/api/v1/classify',
    json={'text': clinical_text}, timeout=30.0).json()
elapsed = int((time.monotonic() - start) * 1000)

print(f'Classification: {result["classification"]}')
print(f'Model: {result["model"]}')
print(f'Accelerator: {result["accelerator"]}')
print(f'Latency: {result["inference_ms"]}ms')
print(f'\n→ Classified as "{result["classification"]}" in {result["inference_ms"]}ms on Xeon 6 CPU')

In [ ]:
# Entity extraction
entities = httpx.post(f'{HEALTHCARE}/api/v1/extract-entities',
    json={'text': clinical_text}, timeout=30.0).json()

print(f'Extracted {len(entities["entities"])} entities in {entities["inference_ms"]}ms')
for e in entities['entities'][:10]:
    print(f'  [{e["type"]:12s}] {e["text"]}')

## 3. Financial Transaction Scoring

The FinServ agent scores transactions for fraud using rule-based signals + LLM reasoning — all on Xeon 6.

In [ ]:
from synthetic.finserv_generator import generate_batch

transactions = generate_batch(10, suspicious_ratio=0.3)
results = []

for tx in transactions:
    r = httpx.post(f'{FINSERV}/api/v1/score-transaction',
        json={'transaction': tx}, timeout=10.0).json()
    results.append(r)
    flag = '⚠' if r['risk_score'] > 30 else '✓'
    print(f'  {flag} ${tx["amount"]:>10,.2f} {tx["type"]:8s} risk={r["risk_score"]:>5.1f} {r["risk_level"]:8s}')

print(f'\n→ 10 transactions scored on Xeon 6 CPU')

## 4. Cross-Agent Workflow

The Go orchestrator dispatches a `patient_financial_risk` workflow that calls both agents via A2A protocol.

In [ ]:
# Start workflow
wf = httpx.post(f'{ORCHESTRATOR}/api/v1/workflows', json={
    'workflow_type': 'patient_financial_risk',
    'input': {'text': clinical_text, 'customer_id': 'cust-notebook-001'}
}, timeout=10.0).json()
print(f'Workflow {wf["id"]} started (status: {wf["status"]})')

# Wait for completion
import time; time.sleep(12)

workflows = httpx.get(f'{ORCHESTRATOR}/api/v1/workflows').json()
for wf in workflows[-1:]:
    print(f'\nWorkflow {wf["id"]}: {wf["status"]} ({wf.get("duration_ms", 0)}ms)')
    print(f'Agents: {wf.get("agents_involved", [])}')
    for step in wf.get('steps', []):
        print(f'  → {step["agent"]}: {step["status"]} ({step.get("inference_ms", 0)}ms)')

## 5. Cost Analysis

How does Xeon 6 self-hosted compare to GPU hardware and cloud APIs?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0A1628'
matplotlib.rcParams['axes.facecolor'] = '#132337'
matplotlib.rcParams['text.color'] = '#E8F0FE'
matplotlib.rcParams['axes.labelcolor'] = '#8CA0B3'
matplotlib.rcParams['xtick.color'] = '#8CA0B3'
matplotlib.rcParams['ytick.color'] = '#8CA0B3'

platforms = ['Xeon 6', 'A100', 'MI300X', 'H100', 'Haiku', 'Gemini', 'Sonnet', 'Opus']
costs = [15000, 64000, 83333, 119333, 10080, 18000, 30240, 50400]
colors = ['#00AEEF', '#76B900', '#ED1C24', '#76B900', '#D4A574', '#4285F4', '#D4A574', '#9B59B6']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(platforms, costs, color=colors, height=0.6)
ax.set_xlabel('Annual Cost (USD) — 100K records/month')
ax.set_title('Infrastructure Cost Comparison', fontsize=14, fontweight='bold', color='#E8F0FE')

for bar, cost in zip(bars, costs):
    ax.text(bar.get_width() + 1500, bar.get_y() + bar.get_height()/2,
            f'${cost:,}', va='center', fontsize=10, color='#E8F0FE')

ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#333')
ax.spines['left'].set_color('#333')
plt.tight_layout()
plt.savefig('../assets/cost-comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Chart saved to demo/assets/cost-comparison.png')

## 6. Inference Telemetry

Every inference call is logged to PostgreSQL with model, latency, and accelerator.

In [ ]:
stats = httpx.get(f'{HEALTHCARE}/api/v1/stats', timeout=5.0).json()

print(f'Total inference calls:  {stats.get("total_requests", 0)}')
print(f'Average latency:       {stats.get("avg_latency_ms", 0):.0f}ms')
print(f'P95 latency:           {stats.get("p95_latency_ms", 0):.0f}ms')
print(f'CPU requests:          {stats.get("cpu_requests", 0)} (100%)')
print(f'GPU requests:          {stats.get("gpu_requests", 0)} (0%)')

calls = stats.get('total_requests', 0)
opus_cost = calls * 0.042
print(f'\n💰 Xeon 6 cost: $0.00')
print(f'💸 Same work on Claude Opus would cost: ${opus_cost:.2f}')
print(f'✅ Saved: ${opus_cost:.2f}')

## 7. The Triforce Value

| Pillar | Technology | What It Proves |
|--------|-----------|----------------|
| **Power** (Intel) | Xeon 6 + AMX via MAAS/LiteLLM | CPU inference handles 80% of enterprise AI — no GPU needed |
| **Wisdom** (IBM) | Kagenti + A2A + MCP + SPIFFE | Agents discover, communicate, and scale with K8s governance |
| **Courage** (Red Hat) | OpenShift + Kafka + PostgreSQL | Enterprise platform runs polyglot agents at production scale |

**Key numbers:**
- $15,000/year — Xeon 6 self-hosted inference
- $489,000/year savings vs Claude Opus at 1M records/month
- 600ms classification latency on CPU
- 246+ tests across Python, Go, Java, TypeScript
- 100% CPU inference — zero GPU required

---
*GitHub: [rhpds/triforce](https://github.com/rhpds/triforce)*